In [3]:
"""
CNN feature extraction + SVM classifier
"""
import time
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from torchvision import transforms, models
import numpy as np
import torch
from torchvision.models import resnet18, ResNet18_Weights

# Timer start
start_time = time.time()

# Load the dataset
ds = load_dataset("mertcobanov/animals")

# Define the image transformation pipeline
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images
    transforms.ToTensor(),         # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize
])

# Feature extraction function
def extract_features(dataset):
    features = []
    labels = []
    
    # Use the updated weights parameter for ResNet18
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)
    model = torch.nn.Sequential(*(list(model.children())[:-1]))  # Remove the last layer
    model.eval()
    
    with torch.no_grad():
        for item in dataset:
            image = transform(item['image']).unsqueeze(0)  # Preprocess the image
            feature = model(image).flatten().numpy()      # Extract features
            features.append(feature)
            labels.append(item['label'])
    return np.array(features), np.array(labels)

# Extract features from the training data
feature_start_time = time.time()
train_features, train_labels = extract_features(ds['train'])
feature_end_time = time.time()

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    train_features, train_labels, test_size=0.2, random_state=42
)

# Train the SVM classifier
svm_start_time = time.time()
svm = SVC(kernel='linear', C=1.0, random_state=42)  # Linear kernel
svm.fit(X_train, y_train)
svm_end_time = time.time()

# Test the model
y_pred = svm.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2%}")

# Print elapsed times
print(f"Feature extraction time: {feature_end_time - feature_start_time:.2f} seconds")
print(f"SVM training time: {svm_end_time - svm_start_time:.2f} seconds")
print(f"Total execution time: {time.time() - start_time:.2f} seconds")



Resolving data files:   0%|          | 0/5400 [00:00<?, ?it/s]

Model Accuracy: 91.11%
Feature extraction time: 95.24 seconds
SVM training time: 1.81 seconds
Total execution time: 99.94 seconds


In [5]:
"""
CNN feature extraction + SVM classifier + Basic transformations such as rotations, flips, cropping, and color jittering.
"""
import time
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from torchvision import transforms, models
import numpy as np
import torch
from torchvision.models import resnet18, ResNet18_Weights

# Timer start
start_time = time.time()

# Load the dataset
ds = load_dataset("mertcobanov/animals")

# Define the image transformation pipeline with data augmentation
transform = transforms.Compose([
    transforms.RandomRotation(degrees=15),       # Randomly rotate ±15 degrees
    transforms.RandomHorizontalFlip(p=0.5),     # Random horizontal flip with 50% probability
    transforms.ColorJitter(brightness=0.2,      # Random brightness adjustment
                           contrast=0.2, 
                           saturation=0.2, 
                           hue=0.1),            # Random hue adjustment
    transforms.RandomResizedCrop(size=(224, 224), scale=(0.8, 1.0)),  # Random crop and resize
    transforms.ToTensor(),                       # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize
])

# Feature extraction function
def extract_features(dataset):
    features = []
    labels = []
    
    # Use the updated weights parameter for ResNet18
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)
    model = torch.nn.Sequential(*(list(model.children())[:-1]))  # Remove the last layer
    model.eval()
    
    with torch.no_grad():
        for item in dataset:
            image = transform(item['image']).unsqueeze(0)  # Preprocess the image
            feature = model(image).flatten().numpy()      # Extract features
            features.append(feature)
            labels.append(item['label'])
    return np.array(features), np.array(labels)

# Extract features from the training data
feature_start_time = time.time()
train_features, train_labels = extract_features(ds['train'])
feature_end_time = time.time()

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    train_features, train_labels, test_size=0.2, random_state=42
)

# Train the SVM classifier
svm_start_time = time.time()
svm = SVC(kernel='linear', C=1.0, random_state=42)  # Linear kernel
svm.fit(X_train, y_train)
svm_end_time = time.time()

# Test the model
y_pred = svm.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2%}")

# Print elapsed times
print(f"Feature extraction time: {feature_end_time - feature_start_time:.2f} seconds")
print(f"SVM training time: {svm_end_time - svm_start_time:.2f} seconds")
print(f"Total execution time: {time.time() - start_time:.2f} seconds")


Resolving data files:   0%|          | 0/5400 [00:00<?, ?it/s]

Model Accuracy: 86.67%
Feature extraction time: 232.72 seconds
SVM training time: 1.98 seconds
Total execution time: 237.16 seconds


In [7]:
"""
CNN feature extraction + SVM classifier + CutMix
"""
import time
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from torchvision import transforms, models
import numpy as np
import torch
from torchvision.models import resnet18, ResNet18_Weights
import random

# Timer start
start_time = time.time()

# Load the dataset
ds = load_dataset("mertcobanov/animals")

# Define the image transformation pipeline (basic preprocessing)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images
    transforms.ToTensor(),         # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize
])

# CutMix augmentation function for classification
def cutmix(image1, image2, label1, label2, alpha=1.0):
    """Applies CutMix to a pair of images and assigns a single label."""
    # Generate lambda from beta distribution
    lam = np.random.beta(alpha, alpha)
    
    # Get image dimensions
    h, w = image1.shape[1], image1.shape[2]
    
    # Determine bounding box coordinates
    cx = np.random.randint(w)
    cy = np.random.randint(h)
    cut_w = int(w * np.sqrt(1 - lam))
    cut_h = int(h * np.sqrt(1 - lam))
    
    x1 = np.clip(cx - cut_w // 2, 0, w)
    x2 = np.clip(cx + cut_w // 2, 0, w)
    y1 = np.clip(cy - cut_h // 2, 0, h)
    y2 = np.clip(cy + cut_h // 2, 0, h)
    
    # Apply CutMix
    mixed_image = image1.clone()
    mixed_image[:, y1:y2, x1:x2] = image2[:, y1:y2, x1:x2]
    
    # Assign the label based on lambda
    mixed_label = label1 if lam > 0.5 else label2
    
    return mixed_image, mixed_label

# Feature extraction function with CutMix
def extract_features_with_cutmix(dataset):
    features = []
    labels = []
    
    # Use the updated weights parameter for ResNet18
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)
    model = torch.nn.Sequential(*(list(model.children())[:-1]))  # Remove the last layer
    model.eval()
    
    with torch.no_grad():
        for i in range(len(dataset)):
            # Apply transformations
            image1 = transform(dataset[i]['image'])
            label1 = dataset[i]['label']
            
            # Choose a random second image for CutMix
            j = random.randint(0, len(dataset) - 1)
            image2 = transform(dataset[j]['image'])
            label2 = dataset[j]['label']
            
            # Apply CutMix augmentation
            mixed_image, mixed_label = cutmix(image1, image2, label1, label2)
            
            # Extract features
            feature = model(mixed_image.unsqueeze(0)).flatten().numpy()
            features.append(feature)
            labels.append(mixed_label)  # Use single label
            
    return np.array(features), np.array(labels)

# Extract features from the training data
feature_start_time = time.time()
train_features, train_labels = extract_features_with_cutmix(ds['train'])
feature_end_time = time.time()

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    train_features, train_labels, test_size=0.2, random_state=42
)

# Train the SVM classifier
svm_start_time = time.time()
svm = SVC(kernel='linear', C=1.0, random_state=42)  # Linear kernel
svm.fit(X_train, y_train)
svm_end_time = time.time()

# Test the model
y_pred = svm.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2%}")

# Print elapsed times
print(f"Feature extraction time: {feature_end_time - feature_start_time:.2f} seconds")
print(f"SVM training time: {svm_end_time - svm_start_time:.2f} seconds")
print(f"Total execution time: {time.time() - start_time:.2f} seconds")


Resolving data files:   0%|          | 0/5400 [00:00<?, ?it/s]

Model Accuracy: 52.04%
Feature extraction time: 296.65 seconds
SVM training time: 3.23 seconds
Total execution time: 303.01 seconds


In [11]:
"""
CNN feature extraction + SVM classifier + Mixup
"""
import time
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from torchvision import transforms, models
import numpy as np
import torch
from torchvision.models import resnet18, ResNet18_Weights
import random

# Timer start
start_time = time.time()

# Load the dataset
ds = load_dataset("mertcobanov/animals")

# Define the image transformation pipeline (basic preprocessing)
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images
    transforms.ToTensor(),         # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize
])

# Mixup augmentation function
def mixup(image1, image2, label1, label2, alpha=1.0):
    """Applies Mixup to a pair of images and assigns a single label."""
    # Generate lambda from beta distribution
    lam = np.random.beta(alpha, alpha)
    
    # Mix images
    mixed_image = lam * image1 + (1 - lam) * image2
    
    # Assign label based on lambda
    mixed_label = label1 if lam > 0.5 else label2  # Use hard labels for classification
    
    return mixed_image, mixed_label

# Feature extraction function with Mixup
def extract_features_with_mixup(dataset):
    features = []
    labels = []
    
    # Use the updated weights parameter for ResNet18
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)
    model = torch.nn.Sequential(*(list(model.children())[:-1]))  # Remove the last layer
    model.eval()
    
    with torch.no_grad():
        for i in range(len(dataset)):
            # Apply transformations
            image1 = transform(dataset[i]['image'])
            label1 = dataset[i]['label']
            
            # Choose a random second image for Mixup
            j = random.randint(0, len(dataset) - 1)
            image2 = transform(dataset[j]['image'])
            label2 = dataset[j]['label']
            
            # Apply Mixup augmentation
            mixed_image, mixed_label = mixup(image1, image2, label1, label2)
            
            # Extract features
            feature = model(mixed_image.unsqueeze(0)).flatten().numpy()
            features.append(feature)
            labels.append(mixed_label)  # Use hard labels
        
    return np.array(features), np.array(labels)

# Extract features from the training data
feature_start_time = time.time()
train_features, train_labels = extract_features_with_mixup(ds['train'])
feature_end_time = time.time()

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    train_features, train_labels, test_size=0.2, random_state=42
)

# Train the SVM classifier
svm_start_time = time.time()
svm = SVC(kernel='linear', C=1.0, random_state=42)  # Linear kernel
svm.fit(X_train, y_train)
svm_end_time = time.time()

# Test the model
y_pred = svm.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2%}")

# Print elapsed times
print(f"Feature extraction time: {feature_end_time - feature_start_time:.2f} seconds")
print(f"SVM training time: {svm_end_time - svm_start_time:.2f} seconds")
print(f"Total execution time: {time.time() - start_time:.2f} seconds")



Resolving data files:   0%|          | 0/5400 [00:00<?, ?it/s]

Model Accuracy: 75.56%
Feature extraction time: 194.62 seconds
SVM training time: 2.27 seconds
Total execution time: 199.22 seconds


In [4]:
"""
CNN feature extraction + SVM classifier + AutoAugment
"""
import time
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from torchvision import transforms, models
from torchvision.transforms import AutoAugment, AutoAugmentPolicy
import numpy as np
import torch
from torchvision.models import resnet18, ResNet18_Weights

# Timer start
start_time = time.time()

# Load the dataset
ds = load_dataset("mertcobanov/animals")

# Define the image transformation pipeline with AutoAugment
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize images
    AutoAugment(policy=AutoAugmentPolicy.IMAGENET),  # Apply AutoAugment with ImageNet policy
    transforms.ToTensor(),         # Convert to tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize
])

# Feature extraction function
def extract_features(dataset):
    features = []
    labels = []
    
    # Use the updated weights parameter for ResNet18
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)
    model = torch.nn.Sequential(*(list(model.children())[:-1]))  # Remove the last layer
    model.eval()
    
    with torch.no_grad():
        for item in dataset:
            # Apply transformations
            image = transform(item['image'])
            label = item['label']
            
            # Extract features
            feature = model(image.unsqueeze(0)).flatten().numpy()
            features.append(feature)
            labels.append(label)
    
    return np.array(features), np.array(labels)

# Extract features from the training data
feature_start_time = time.time()
train_features, train_labels = extract_features(ds['train'])
feature_end_time = time.time()

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    train_features, train_labels, test_size=0.2, random_state=42
)

# Train the SVM classifier
svm_start_time = time.time()
svm = SVC(kernel='linear', C=1.0, random_state=42)  # Linear kernel
svm.fit(X_train, y_train)
svm_end_time = time.time()

# Test the model
y_pred = svm.predict(X_test)

# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy:.2%}")

# Print elapsed times
print(f"Feature extraction time: {feature_end_time - feature_start_time:.2f} seconds")
print(f"SVM training time: {svm_end_time - svm_start_time:.2f} seconds")
print(f"Total execution time: {time.time() - start_time:.2f} seconds")


Resolving data files:   0%|          | 0/5400 [00:00<?, ?it/s]

Model Accuracy: 72.31%
Feature extraction time: 103.21 seconds
SVM training time: 2.42 seconds
Total execution time: 108.20 seconds
